In [0]:
SELECT
  id,
  enriched_text,
  ai_query(
    'databricks-meta-llama-3-3-70b-instruct',
    concat('''
    You are an expert in geothermal energy projects. 
    Classify the following text chunk into one of these categories, using the descriptions to guide your choice:

    - Geoscience: Geology, subsurface characterization, reservoir properties, rock/mineral analysis, geophysical or geochemical data, resource assessment.
    - Drilling & Well Engineering: Well design, drilling operations, completions, casing, cementing, well integrity, drilling equipment.
    - Surface Facilities & Power Plant: Surface infrastructure, power plant design, turbines, heat exchangers, cooling systems, surface piping.
    - Operations & Maintenance: Monitoring, maintenance, troubleshooting, operational procedures, performance optimization, safety.
    - Economics & Policy: Project economics, cost analysis, financing, regulatory issues, permitting, incentives, market analysis.
    - Environmental & Social: Environmental impact, permitting, community engagement, land use, water management, emissions.
    - Other: Does not fit any of the above.

    Respond ONLY with a valid JSON object containing the key "classification" and the value as the chosen category.
    Example: {"classification": "Geoscience"}

    Text:
    ''', enriched_text),
    responseFormat => '{
        "type": "json_schema",
        "json_schema": {
          "name": "geothermal_classification",
          "schema": {
            "type": "object",
            "properties": {
              "classification": {
                "type": "string",
                "enum": [
                  "Geoscience",
                  "Drilling & Well Engineering",
                  "Surface Facilities & Power Plant",
                  "Operations & Maintenance",
                  "Economics & Policy",
                  "Environmental & Social",
                  "Other"
                ]
              }
            },
            "required": ["classification"]
          }
        }
      }'
  ) AS classified_clause
FROM chunks
WHERE chunk_type = 'text'
LIMIT 3 -- remove for production